# Importing dependencies

In [1]:
# Dependencies (Monte Carlo contingency testing requires SciPy >= 1.15)
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import MonteCarloMethod, chi2_contingency

# Loading CSV

In [2]:
# Run the notebook from the project folder.
data_path = Path("hcvdat0.csv")
raw_df = pd.read_csv(data_path)
raw_df.head()

,Unnamed: 0,Category,Age,Sex,ALB,ALP,ALT,AST,BIL,CHE,CHOL,CREA,GGT,PROT
0,1,0=Blood Donor,32,m,38.5,52.5,7.7,22.1,7.5,6.93,3.23,106.0,12.1,69.0
1,2,0=Blood Donor,32,m,38.5,70.3,18.0,24.7,3.9,11.17,4.80,74.0,15.6,76.5
2,3,0=Blood Donor,32,m,46.9,74.7,36.2,52.6,6.1,8.84,5.20,86.0,33.2,79.3
3,4,0=Blood Donor,32,m,43.2,52.0,30.6,22.6,18.9,7.33,4.74,80.0,33.8,75.7
4,5,0=Blood Donor,32,m,39.2,74.1,32.6,24.8,9.6,9.15,4.32,76.0,29.9,68.7


# Data Cleaning and EDA stage

In [3]:
# Remove the patient ID and clean the category labels.
df = raw_df.drop(columns=["Unnamed: 0"], errors="ignore").copy()
df["Category"] = df["Category"].str.replace(r"^[^=]*=", "", regex=True)

category_codes = {
    "Blood Donor": 0,
    "suspect Blood Donor": 1,
    "Suspected Blood Donor": 1,
    "Hepatitis": 2,
    "Fibrosis": 3,
    "Cirrhosis": 4,
}
df["cat_num"] = df["Category"].map(category_codes).astype("Int64")
df["Category"] = df["Category"].replace(
    {"suspect Blood Donor": "Suspected Blood Donor"}
)
print(f"Rows: {df.shape[0]}, columns: {df.shape[1]}")
df.head()

Rows: 615, columns: 14


,Category,Age,Sex,ALB,ALP,ALT,AST,BIL,CHE,CHOL,CREA,GGT,PROT,cat_num
0,Blood Donor,32,m,38.5,52.5,7.7,22.1,7.5,6.93,3.23,106.0,12.1,69.0,0
1,Blood Donor,32,m,38.5,70.3,18.0,24.7,3.9,11.17,4.80,74.0,15.6,76.5,0
2,Blood Donor,32,m,46.9,74.7,36.2,52.6,6.1,8.84,5.20,86.0,33.2,79.3,0
3,Blood Donor,32,m,43.2,52.0,30.6,22.6,18.9,7.33,4.74,80.0,33.8,75.7,0
4,Blood Donor,32,m,39.2,74.1,32.6,24.8,9.6,9.15,4.32,76.0,29.9,68.7,0


In [ ]:
# Show each category, its code and its number of patients.
category_summary = (
    df.groupby(["cat_num", "Category"], dropna=False)
      .size()
      .reset_index(name="Count")
      .sort_values("cat_num")
)
category_summary

,cat_num,Category,Count
0,0,Blood Donor,533
1,1,Suspected Blood Donor,7
2,2,Hepatitis,24
3,3,Fibrosis,21
4,4,Cirrhosis,30


In [5]:
# Count missing values and calculate their percentage in each column.
missing_summary = pd.DataFrame({
    "Missing": df.isna().sum(),
    "Missing (%)": df.isna().mean().mul(100).round(2),
})
display(missing_summary)

,Missing,Missing (%)
Category,0,0.00
Age,0,0.00
Sex,0,0.00
ALB,1,0.16
ALP,18,2.93
ALT,1,0.16
AST,0,0.00
BIL,0,0.00
CHE,0,0.00
CHOL,10,1.63


In [6]:
# Missing percentages within each category, before any imputation.
missing_columns = missing_summary.index[missing_summary["Missing"] > 0].tolist()
missing_by_category = (
    df[missing_columns].isna()
      .groupby(df["Category"])
      .mean().mul(100).round(2)
)
display(missing_by_category)

,ALB,ALP,ALT,CHOL,PROT
Category,,,,,
Blood Donor,0.00,0.00,0.00,1.31,0.00
Cirrhosis,3.33,20.00,0.00,6.67,3.33
Fibrosis,0.00,42.86,0.00,4.76,0.00
Hepatitis,0.00,12.50,4.17,0.00,0.00
Suspected Blood Donor,0.00,0.00,0.00,0.00,0.00


In [ ]:
# 1. Set up the analysis.
# Use 10,000 resamples and a fixed seed for reproducible estimates.
category_columns = ["Category", "cat_num"]
number_of_resamples = 10000
random_seed = 42
significance_level = 0.05

# Compare patients only when their disease category is known.
known_category_data = df.loc[df[category_columns].notna()]

# Test incomplete measurements, excluding the diagnosis and its numeric code.
columns_to_test = []
for column in missing_columns:
    if column not in category_columns:
        columns_to_test.append(column)

# 2. Test each measurement separately.
test_results = []
for column in columns_to_test:
    # isna() labels each value as missing (True) or present (False).
    # crosstab() counts these two groups within each disease category.
    missing_flags = known_category_data[column].isna()
    count_table = pd.crosstab(known_category_data[category_columns], missing_flags)

    # Without variation in both groups, an association cannot be tested.
    has_multiple_categories = count_table.shape[0] >= 2
    has_missing_and_present = count_table.shape[1] >= 2
    p_value = np.nan

    if has_multiple_categories and has_missing_and_present:
        # Monte Carlo sampling avoids the large-count chi-square approximation.
        # Restart the fixed seed for each column for reproducibility.
        sampling_method = MonteCarloMethod(
            n_resamples=number_of_resamples,
            rng=np.random.default_rng(random_seed),
        )
        test = chi2_contingency(
            count_table, correction=False, method=sampling_method
        )
        p_value = test.pvalue

    test_results.append({
        "Column": column,
        "Missing": int(df[column].isna().sum()),
        "p_value": p_value,
    })

missingness_results = pd.DataFrame(
    test_results, columns=["Column", "Missing", "p_value"]
)

# 3. Adjust for testing several measurements (Bonferroni correction).
# Multiply each p-value by the number of valid tests; p-values cannot exceed 1.
number_of_tests = missingness_results["p_value"].notna().sum()
missingness_results["adjusted_p"] = (
    missingness_results["p_value"] * number_of_tests
).clip(upper=1)

# 4. Explain each adjusted p-value.
# A small p-value suggests association; a large one does NOT prove randomness.
interpretations = []
for adjusted_p in missingness_results["adjusted_p"]:
    if pd.isna(adjusted_p):
        interpretation = "Not testable"
    elif adjusted_p < significance_level:
        interpretation = "Evidence of association with category"
    else:
        interpretation = "No association detected; randomness NOT proven"
    interpretations.append(interpretation)

missingness_results["Interpretation"] = interpretations

# 5. Show the results and the limits of this check.
print("This checks association with Category; it cannot prove randomness or distinguish MAR from MNAR.")
missingness_results


This checks association with Category; it cannot prove randomness or distinguish MAR from MNAR.


,Column,Missing,p_value,adjusted_p,Interpretation
0,ALB,1,0.133087,0.665433,No association detected; randomness NOT proven
1,ALP,18,0.000100,0.000500,Evidence of association with category
2,ALT,1,0.083492,0.417458,No association detected; randomness NOT proven
3,CHOL,10,0.217178,1.000000,No association detected; randomness NOT proven
4,PROT,1,0.133087,0.665433,No association detected; randomness NOT proven


### Missing-value findings

Five columns contain missing values: ALB (1), ALP (18), ALT (1), CHOL (10) and PROT (1). The Monte Carlo chi-square tests (10,000 resamples, fixed seed) compared missingness across disease categories, with a Bonferroni adjustment for the five comparisons.

| Column | Missing values | Adjusted p-value | Finding |
|---|---:|---:|---|
| ALB | 1 | 0.6654 | No association detected |
| ALP | 18 | 0.0005 | Evidence of association with disease category |
| ALT | 1 | 0.4175 | No association detected |
| CHOL | 10 | 1.0000 | No association detected |
| PROT | 1 | 0.6654 | No association detected |

**ALP shows the clearest pattern.** It is missing in 42.86% of fibrosis patients (9/21), 20% of cirrhosis patients (6/30) and 12.5% of hepatitis patients (3/24), compared with 0% in both donor groups. Although only 2.9% of ALP values are missing overall, the missingness is concentrated in patients with liver disease. This provides evidence against ALP being missing completely at random (MCAR).

For the other columns, no statistically significant association was detected after adjustment. This does **not** prove their missingness is random. ALB, ALT and PROT each have only one missing value, which limits how much the tests can tell us. These checks only examine disease category and cannot distinguish missing at random (MAR) from missing not at random (MNAR).

**Implication:** dropping rows with missing values could disproportionately remove patients with liver disease, especially fibrosis. Imputation methods should therefore be compared during model validation rather than chosen from the overall missing percentage alone. Replacement values must be learned from training data only; disease category must not be used to fill predictors when it is the outcome being predicted.

The smallest raw Monte Carlo p-value with 10,000 resamples is 1/10,001 (approximately 0.0001). ALP reaches this limit, so its adjusted value of approximately 0.0005 reflects simulation resolution rather than a precise estimate of the underlying p-value. The association remains significant.
